In [0]:
%run ./_utils

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

#### Create `openalex.awards.openalex_awards_snapshot` in same format as API

In [ ]:
df_transformed = (
    spark.read.table("openalex.awards.awards_api")
    # Convert BIGINT id to full URL
    .withColumn("id", F.concat(F.lit("https://openalex.org/G"), F.col("id")))
    # Coalesce null arrays to empty arrays
    .withColumn("investigators", F.coalesce(F.col("investigators"), F.array()))
    .withColumn("funded_outputs", F.coalesce(F.col("funded_outputs"), F.array()))
    # Drop internal funder_id (already captured inside funder struct)
    .drop("funder_id")
)

df_transformed.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("openalex.awards.openalex_awards_snapshot")

#### Export to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/awards/`.

In [0]:
date_str = get_snapshot_date()
df = spark.read.table("openalex.awards.openalex_awards_snapshot")
export_partitioned_all_formats(spark, dbutils, df, date_str, "awards", salt=False)